In [ ]:
import cv2
import json
from mtcnn import MTCNN
from tensorflow.keras.models import load_model
import numpy as np
from numpy.linalg import norm
import matplotlib.pyplot as plt
from tensorflow.keras import backend as K # Import backend
# Define scaling function (You might need to adjust this according to how it was used in your model)
def scaling(x, scale):
    return x * scale

# Define l2_normalize function
def l2_normalize(x, axis=-1, epsilon=1e-12):
    """
    L2 normalization of the input tensor along the specified axis.
    """
    square_sum = K.sum(K.square(x), axis=axis, keepdims=True)
    x_inv_norm = K.sqrt(K.maximum(square_sum, epsilon))
    return x / x_inv_norm


# Charger le modèle FaceNet
model = load_model('FaceNetclassification_model1.h5', custom_objects={'scaling': scaling, 'l2_normalize': l2_normalize})

# Initialiser MTCNN pour détecter les visages
detector = MTCNN()

# Charger ou créer la base de données des embeddings
try:
    with open('face_embeddings_database1.json', 'r') as f:
        database_embeddings = json.load(f)
except FileNotFoundError:
    database_embeddings = {}

# Fonction pour générer l'embedding d'un visage
def get_embedding(model, face_pixels):
    # Redimensionner l'image à la taille attendue (160x160 pour FaceNet)
    face_pixels = cv2.resize(face_pixels, (160, 160))

    # Normaliser les pixels
    face_pixels = face_pixels.astype('float32')
    mean, std = face_pixels.mean(), face_pixels.std()
    face_pixels = (face_pixels - mean) / std

    # Ajouter une dimension pour correspondre aux attentes du modèle
    face_pixels = np.expand_dims(face_pixels, axis=0)

    # Obtenir l'embedding
    embedding = model.predict(face_pixels)
    return embedding

# Fonction pour aligner le visage selon les yeux
def align_face(image, keypoints):
    left_eye = keypoints['left_eye']
    right_eye = keypoints['right_eye']

    # Calculate the center of the eyes using floating-point division
    # This ensures the coordinates are of the correct type (float)
    eyes_center = ((left_eye[0] + right_eye[0]) / 2, (left_eye[1] + right_eye[1]) / 2)

    dy = right_eye[1] - left_eye[1]
    dx = right_eye[0] - left_eye[0]
    angle = np.degrees(np.arctan2(dy, dx))

    # Convert eyes_center to a tuple of integers
    eyes_center = (int(eyes_center[0]), int(eyes_center[1]))

    rot_mat = cv2.getRotationMatrix2D(eyes_center, angle, 1)
    aligned_face = cv2.warpAffine(image, rot_mat, (image.shape[1], image.shape[0]))

    return aligned_face

# Ajouter des images et leurs embeddings à la base de données
def add_embedding_to_database(image_path, person_name):
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    faces = detector.detect_faces(image_rgb)

    if len(faces) > 0:
        face = faces[0]  # Prenez le premier visage détecté
        bounding_box = face['box']
        keypoints = face['keypoints']

        # Extraire et aligner le visage
        x, y, w, h = bounding_box
        face_crop = image_rgb[y:y+h, x:x+w]
        aligned_face = align_face(face_crop, keypoints)

        # Obtenir l'embedding
        embedding = get_embedding(model, aligned_face)

        # Ajouter l'embedding à la base de données
        database_embeddings[person_name] = embedding.tolist()
        print(f"Embedding ajouté pour {person_name}")

    else:
        print(f"Aucun visage détecté dans {image_path}")

# Ajouter plusieurs images à la base de données
image_paths = {
    "Amal Bouabid": "1717074356030.jpg",
    "Ronaldo": "ro.png",
    "Krysten Ritter": "Krysten Ritter4_1822.jpg",
    "Ichrak Aguir": "WhatsApp Image 2024-12-08 at 11.53.13.jpeg",
     "Aouadhi Moez": "WhatsApp Image 2024-12-08 at 22.03.07 (5).jpeg"


}

# Ajouter les embeddings de chaque image
for person_name, image_path in image_paths.items():
    add_embedding_to_database(image_path, person_name)

# Sauvegarder la base de données des embeddings dans un fichier JSON
with open('face_embeddings_database.json', 'w') as f:
    json.dump(database_embeddings, f)

print("Base de données des embeddings mise à jour et sauvegardée.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step
Embedding ajouté pour Amal Bouabid
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
Embedding ajouté pour Ronaldo
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
Embedding ajouté pour Krysten Ritter
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Embedding ajouté pour Ichrak Aguir
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Embedding ajouté pour Aouadhi Moez
Base de données des embeddings mise à jour et sauvegardée.
